# Building Event Models

This notebook demonstrates how to build event models for fMRI analysis using pyfmridesign.

Author: Bradley R. Buchsbaum (Python port)  
Date: 2024

## Introduction: Modeling Task Effects

An **event model** describes the expected BOLD signal changes related to experimental events (stimuli, conditions, responses, etc.). It forms the core of the task-related component of an fMRI General Linear Model (GLM).

`pyfmridesign` primarily uses the `event_model()` function to create these models. It takes experimental design information (event onsets, conditions, durations) and combines it with Hemodynamic Response Function (HRF) specifications to generate the task regressors.

This notebook focuses on the formula-based interface for `event_model()`, which provides a flexible and intuitive way to specify complex designs.

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from pyfmridesign import (
    event_model, EventFactor, EventVariable,
    HRF_SPMG1, gen_hrf, SamplingFrame,
    design_matrix, design_map, baseline_model,
    contrast, contrast_weights, Poly
)

# Set up plotting style
plt.style.use('seaborn-v0_8-darkgrid')

## A Simple Single-Factor Design (Single Run)

Consider a basic design with four stimulus types (*face*, *scene*, *tool*, *object*), each presented 4 times for 2s, separated by a variable ISI (4-7s). The total scan duration is 70 TRs (TR=2s).

In [ ]:
# Set up design parameters
TR = 2.0
conditions = ["face", "scene", "tool", "object"]
n_trials_per_cond = 4
n_trials_total = len(conditions) * n_trials_per_cond

# Create trial sequence (randomized)
np.random.seed(42)
trial_sequence = np.repeat(conditions, n_trials_per_cond)
np.random.shuffle(trial_sequence)

# Generate onsets with variable ISI
isi_values = np.random.uniform(4, 7, n_trials_total)
stimulus_duration = 2.0
onsets = np.cumsum(np.concatenate([[5], isi_values[:-1] + stimulus_duration]))

# Create design dataframe
design_df = pd.DataFrame({
    'onset': onsets,
    'stimulus': trial_sequence,
    'duration': stimulus_duration
})

print("Design summary:")
print(design_df.head(10))
print(f"\nTotal experiment duration: {onsets[-1] + stimulus_duration:.1f} seconds")

## Creating the Event Model

Now we'll create an event model using the formula interface. The formula `onset ~ stimulus` specifies that we want to model onset times as a function of stimulus type:

In [ ]:
# Create sampling frame
n_scans = 70
sframe = SamplingFrame(n_scans=n_scans, tr=TR)

# Create event model
emodel = event_model(
    "onset ~ stimulus",
    data=design_df,
    sampling_frame=sframe,
    hrf=HRF_SPMG1,
    durations="duration"  # Use the duration column
)

# Extract design matrix
X = design_matrix(emodel)
print(f"Design matrix shape: {X.shape}")
print(f"Conditions modeled: {X.shape[1]} (one per stimulus type)")

# Visualize the design matrix
plt.figure(figsize=(10, 8))
plt.imshow(X, aspect='auto', cmap='RdBu_r', interpolation='nearest')
plt.colorbar(label='Regressor Value')
plt.xlabel('Stimulus Type')
plt.ylabel('Time (scans)')
plt.title('Design Matrix for Single-Factor Design')
plt.xticks(range(4), conditions)
plt.tight_layout()
plt.show()

## Visualizing Model Components

Let's look at the individual regressors and how they relate to the stimulus presentation:

In [ ]:
# Plot each regressor
time_points = np.arange(n_scans) * TR

fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)

for idx, (cond, ax) in enumerate(zip(conditions, axes)):
    # Plot regressor
    ax.plot(time_points, X[:, idx], 'b-', linewidth=2)
    
    # Mark stimulus presentations
    cond_onsets = design_df[design_df['stimulus'] == cond]['onset'].values
    for onset in cond_onsets:
        ax.axvspan(onset, onset + stimulus_duration, 
                   alpha=0.3, color='red', label='Stimulus' if onset == cond_onsets[0] else '')
    
    ax.set_ylabel('Signal')
    ax.set_title(f'{cond.capitalize()} Regressor')
    ax.grid(True, alpha=0.3)
    if idx == 0:
        ax.legend()

axes[-1].set_xlabel('Time (seconds)')
plt.suptitle('Individual Regressors by Condition', fontsize=14)
plt.tight_layout()
plt.show()

## Multi-Factor Design: Factorial Experiment

Let's create a more complex 2x2 factorial design with factors:
- **Category**: animate vs. inanimate
- **Size**: large vs. small

In [ ]:
# Create factorial design
categories = ['animate', 'inanimate']
sizes = ['large', 'small']
n_reps = 5  # repetitions per cell

# Generate all combinations
factorial_design = []
for cat in categories:
    for size in sizes:
        for _ in range(n_reps):
            factorial_design.append({'category': cat, 'size': size})

# Randomize and add timing
np.random.seed(123)
np.random.shuffle(factorial_design)

# Generate onsets
n_trials = len(factorial_design)
isi = np.random.uniform(3, 6, n_trials)
onsets = np.cumsum(np.concatenate([[4], isi[:-1] + 1.5]))  # 1.5s stimulus duration

# Create dataframe
factorial_df = pd.DataFrame(factorial_design)
factorial_df['onset'] = onsets
factorial_df['duration'] = 1.5

print("Factorial design summary:")
print(factorial_df.groupby(['category', 'size']).size())
print(f"\nTotal trials: {len(factorial_df)}")

## Modeling Main Effects and Interactions

We can use formula notation to specify main effects and interactions:

In [ ]:
# Create sampling frame for longer experiment
n_scans_factorial = 120
sframe_factorial = SamplingFrame(n_scans=n_scans_factorial, tr=TR)

# Model with main effects only
emodel_main = event_model(
    "onset ~ category + size",
    data=factorial_df,
    sampling_frame=sframe_factorial,
    hrf=HRF_SPMG1,
    durations="duration"
)

# Model with interaction
emodel_interaction = event_model(
    "onset ~ category * size",  # Equivalent to category + size + category:size
    data=factorial_df,
    sampling_frame=sframe_factorial,
    hrf=HRF_SPMG1,
    durations="duration"
)

# Get design matrices
X_main = design_matrix(emodel_main)
X_interaction = design_matrix(emodel_interaction)

print(f"Main effects model: {X_main.shape[1]} regressors")
print(f"Full factorial model: {X_interaction.shape[1]} regressors")

# Visualize both models
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 8))

# Main effects only
im1 = ax1.imshow(X_main, aspect='auto', cmap='RdBu_r', interpolation='nearest')
ax1.set_title('Main Effects Only')
ax1.set_xlabel('Regressors')
ax1.set_ylabel('Time (scans)')
ax1.set_xticks(range(X_main.shape[1]))
ax1.set_xticklabels(['animate', 'inanimate', 'large', 'small'], rotation=45)

# Full factorial
im2 = ax2.imshow(X_interaction, aspect='auto', cmap='RdBu_r', interpolation='nearest')
ax2.set_title('Full Factorial (with Interaction)')
ax2.set_xlabel('Regressors')
ax2.set_ylabel('Time (scans)')
ax2.set_xticks(range(X_interaction.shape[1]))
ax2.set_xticklabels(['ani.large', 'ani.small', 'inani.large', 'inani.small'], rotation=45)

plt.colorbar(im1, ax=ax1)
plt.colorbar(im2, ax=ax2)
plt.tight_layout()
plt.show()

## Parametric Modulation

Event models can include continuous covariates to model parametric effects:

In [ ]:
# Add reaction time as a continuous covariate
np.random.seed(456)
# Simulate RTs: faster for animate/large, slower for inanimate/small
base_rt = 0.8
rt_effects = {
    ('animate', 'large'): -0.15,
    ('animate', 'small'): -0.05,
    ('inanimate', 'large'): 0.05,
    ('inanimate', 'small'): 0.15
}

rts = []
for _, row in factorial_df.iterrows():
    effect = rt_effects[(row['category'], row['size'])]
    rt = base_rt + effect + np.random.normal(0, 0.1)
    rts.append(max(0.3, rt))  # Floor at 300ms

factorial_df['rt'] = rts

# Create parametric model
emodel_parametric = event_model(
    "onset ~ category * size + rt",
    data=factorial_df,
    sampling_frame=sframe_factorial,
    hrf=HRF_SPMG1,
    durations="duration"
)

X_parametric = design_matrix(emodel_parametric)
print(f"Parametric model: {X_parametric.shape[1]} regressors")

# Visualize the parametric regressor
plt.figure(figsize=(12, 6))

# Plot RT regressor (last column)
time_points = np.arange(n_scans_factorial) * TR
plt.subplot(2, 1, 1)
plt.plot(time_points, X_parametric[:, -1], 'purple', linewidth=2)
plt.ylabel('RT Regressor')
plt.title('Reaction Time Parametric Modulator')
plt.grid(True, alpha=0.3)

# Plot RT values over time
plt.subplot(2, 1, 2)
plt.scatter(factorial_df['onset'], factorial_df['rt'], 
            c=factorial_df['category'].map({'animate': 'red', 'inanimate': 'blue'}),
            s=factorial_df['size'].map({'large': 100, 'small': 40}),
            alpha=0.7)
plt.xlabel('Time (seconds)')
plt.ylabel('Reaction Time (s)')
plt.title('Reaction Times by Condition')
plt.legend(['Animate (red) vs Inanimate (blue)', 'Size: Large (big) vs Small (small)'])
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Multi-run Experiments

Real fMRI experiments often consist of multiple runs. The `event_model` function handles this through the `block` parameter:

In [ ]:
# Create a 3-run experiment
n_runs = 3
trials_per_run = 12

multirun_data = []
for run in range(1, n_runs + 1):
    # Each run has the same structure but different randomization
    run_conditions = np.repeat(['A', 'B', 'C'], trials_per_run // 3)
    np.random.shuffle(run_conditions)
    
    # Generate onsets (local to each run)
    isi = np.random.uniform(4, 8, len(run_conditions))
    onsets = np.cumsum(np.concatenate([[6], isi[:-1] + 2]))
    
    for onset, cond in zip(onsets, run_conditions):
        multirun_data.append({
            'onset': onset,
            'condition': cond,
            'run': run,
            'duration': 2.0
        })

multirun_df = pd.DataFrame(multirun_data)

# Create multi-run sampling frame
scans_per_run = 80
sframe_multirun = SamplingFrame(
    blocklens=[scans_per_run] * n_runs,
    tr=TR
)

# Create event model with run structure
emodel_multirun = event_model(
    "onset ~ condition",
    data=multirun_df,
    block="run",  # Specify the run column
    sampling_frame=sframe_multirun,
    hrf=HRF_SPMG1,
    durations="duration"
)

# Get design matrix
X_multirun = design_matrix(emodel_multirun)

# Add baseline model for visualization
baseline_multirun = baseline_model(sframe_multirun, basis="poly", degree=2)
X_baseline = design_matrix(baseline_multirun)
X_full = np.column_stack([X_multirun, X_baseline])

# Visualize with run structure
fig = plt.figure(figsize=(14, 10))

# Use design_map for better visualization
from pyfmridesign import design_map
ax = design_map(X_full, show=False)

# Add run boundaries
run_boundaries = np.cumsum([0] + [scans_per_run] * n_runs)
for boundary in run_boundaries[1:-1]:
    ax.axhline(y=boundary, color='black', linewidth=2, linestyle='--')

# Add labels
ax.axvline(x=3, color='black', linewidth=2)
ax.text(1.5, -10, 'Task', ha='center', fontsize=12, fontweight='bold')
ax.text(X_full.shape[1] - 3, -10, 'Baseline', ha='center', fontsize=12, fontweight='bold')

# Add run labels
for i in range(n_runs):
    y_pos = run_boundaries[i] + scans_per_run // 2
    ax.text(-1, y_pos, f'Run {i+1}', ha='right', va='center', fontsize=10, fontweight='bold')

plt.title('Multi-run Design Matrix', fontsize=14)
plt.tight_layout()
plt.show()

print(f"Full design matrix shape: {X_full.shape}")
print(f"Task regressors: {X_multirun.shape[1]}")
print(f"Baseline regressors: {X_baseline.shape[1]}")

## Using Different HRF Models

The choice of HRF can significantly impact model fit. Let's compare different HRF specifications:

In [ ]:
# Simple design for comparison
simple_df = pd.DataFrame({
    'onset': [10, 25, 40, 55, 70, 85],
    'condition': ['A', 'B', 'A', 'B', 'A', 'B'],
    'duration': 1.0
})

sframe_simple = SamplingFrame(n_scans=60, tr=TR)

# Different HRF models
hrf_models = {
    'Canonical': HRF_SPMG1,
    'With Derivative': gen_hrf(type="spmg1", include_deriv=1),
    'FIR (8 bins)': gen_hrf(type="fir", nbasis=8, span=16),
    'Gamma (custom)': gen_hrf(type="gamma", shape=8, scale=0.9)
}

# Create models and visualize
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for idx, (name, hrf) in enumerate(hrf_models.items()):
    # Create event model
    emodel_hrf = event_model(
        "onset ~ condition",
        data=simple_df,
        sampling_frame=sframe_simple,
        hrf=hrf,
        durations="duration"
    )
    
    # Get design matrix
    X_hrf = design_matrix(emodel_hrf)
    
    # Plot
    ax = axes[idx]
    im = ax.imshow(X_hrf.T, aspect='auto', cmap='RdBu_r', 
                   interpolation='nearest')
    
    ax.set_title(f'{name} (shape: {X_hrf.shape})')
    ax.set_xlabel('Time (scans)')
    ax.set_ylabel('Regressors')
    
    # Add onset markers
    for onset in simple_df['onset']:
        ax.axvline(x=onset/TR, color='green', linestyle='--', alpha=0.5)
    
    plt.colorbar(im, ax=ax)

plt.suptitle('Event Models with Different HRFs', fontsize=16)
plt.tight_layout()
plt.show()

## Contrasts and Hypothesis Testing

After building the model, we typically want to test specific hypotheses using contrasts:

In [ ]:
# Use the factorial model from earlier
# Define contrasts
contrasts = {
    'animate_vs_inanimate': contrast(
        "category[animate] - category[inanimate]",
        name="Animate > Inanimate"
    ),
    'large_vs_small': contrast(
        "size[large] - size[small]",
        name="Large > Small"
    ),
    'interaction': contrast(
        "(category[animate]:size[large] - category[animate]:size[small]) - "
        "(category[inanimate]:size[large] - category[inanimate]:size[small])",
        name="Category × Size Interaction"
    )
}

# Compute contrast weights
contrast_matrices = {}
for name, con in contrasts.items():
    weights = contrast_weights(con, emodel_interaction)
    contrast_matrices[name] = weights['weights']

# Visualize contrasts
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, (name, weights) in enumerate(contrast_matrices.items()):
    ax = axes[idx]
    
    # Create bar plot
    x_pos = np.arange(len(weights))
    colors = ['green' if w > 0 else 'red' if w < 0 else 'gray' for w in weights]
    ax.bar(x_pos, weights, color=colors, alpha=0.7)
    
    ax.set_xlabel('Conditions')
    ax.set_ylabel('Contrast Weight')
    ax.set_title(contrasts[name].name)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(['ani.large', 'ani.small', 'inani.large', 'inani.small'], 
                       rotation=45)
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Contrast Weights for Factorial Design', fontsize=14)
plt.tight_layout()
plt.show()

# Verify contrast properties
for name, weights in contrast_matrices.items():
    print(f"\n{name}:")
    print(f"  Sum of weights: {np.sum(weights):.6f} (should be ~0)")
    print(f"  Non-zero weights: {np.sum(weights != 0)}")

## Advanced: Mixed Designs with Multiple Event Types

Real experiments often have multiple types of events (e.g., stimuli, responses, feedback):

In [ ]:
# Create a more complex design
n_trials = 20
complex_data = []

for i in range(n_trials):
    # Stimulus onset
    stim_onset = 10 + i * 8
    stim_type = np.random.choice(['word', 'picture'])
    
    # Response (1-3s after stimulus)
    rt = np.random.uniform(1, 3)
    response_onset = stim_onset + rt
    accuracy = np.random.choice(['correct', 'error'], p=[0.85, 0.15])
    
    # Feedback (1s after response)
    feedback_onset = response_onset + 1
    
    # Add all events
    complex_data.extend([
        {'onset': stim_onset, 'event_type': 'stimulus', 
         'modality': stim_type, 'duration': 0.5},
        {'onset': response_onset, 'event_type': 'response', 
         'accuracy': accuracy, 'duration': 0},
        {'onset': feedback_onset, 'event_type': 'feedback', 
         'accuracy': accuracy, 'duration': 0.5}
    ])

complex_df = pd.DataFrame(complex_data)

# Create separate models for each event type
sframe_complex = SamplingFrame(n_scans=200, tr=TR)

# Stimulus model
stim_data = complex_df[complex_df['event_type'] == 'stimulus'].copy()
stim_model = event_model(
    "onset ~ modality",
    data=stim_data,
    sampling_frame=sframe_complex,
    hrf=HRF_SPMG1,
    durations="duration"
)

# Response model (with different HRF)
resp_data = complex_df[complex_df['event_type'] == 'response'].copy()
resp_model = event_model(
    "onset ~ accuracy",
    data=resp_data,
    sampling_frame=sframe_complex,
    hrf=gen_hrf(type="gamma", shape=4, scale=1),  # Earlier, sharper HRF
    durations="duration"
)

# Combine models
X_stim = design_matrix(stim_model)
X_resp = design_matrix(resp_model)
X_combined = np.column_stack([X_stim, X_resp])

# Visualize combined model
plt.figure(figsize=(12, 8))

# Create subplot layout
gs = plt.GridSpec(3, 1, height_ratios=[1, 1, 2])

# Stimulus regressors
ax1 = plt.subplot(gs[0])
im1 = ax1.imshow(X_stim.T, aspect='auto', cmap='Blues', interpolation='nearest')
ax1.set_ylabel('Stimulus')
ax1.set_yticks([0, 1])
ax1.set_yticklabels(['Word', 'Picture'])
ax1.set_xticks([])

# Response regressors
ax2 = plt.subplot(gs[1])
im2 = ax2.imshow(X_resp.T, aspect='auto', cmap='Reds', interpolation='nearest')
ax2.set_ylabel('Response')
ax2.set_yticks([0, 1])
ax2.set_yticklabels(['Correct', 'Error'])
ax2.set_xticks([])

# Combined
ax3 = plt.subplot(gs[2])
im3 = ax3.imshow(X_combined.T, aspect='auto', cmap='RdBu_r', interpolation='nearest')
ax3.set_ylabel('All Regressors')
ax3.set_xlabel('Time (scans)')
ax3.set_yticks(range(4))
ax3.set_yticklabels(['Word', 'Picture', 'Correct', 'Error'])

plt.suptitle('Mixed Event Model: Stimuli and Responses', fontsize=14)
plt.tight_layout()
plt.show()

print(f"Combined design matrix: {X_combined.shape}")
print(f"Stimulus regressors: {X_stim.shape[1]}")
print(f"Response regressors: {X_resp.shape[1]}")

## Summary

In this notebook, we've covered:

1. **Basic event models** - Single-factor designs with categorical predictors
2. **Factorial designs** - Modeling main effects and interactions
3. **Parametric modulation** - Including continuous covariates
4. **Multi-run experiments** - Handling blocked designs
5. **HRF selection** - Impact of different basis functions
6. **Hypothesis testing** - Defining and visualizing contrasts
7. **Complex designs** - Multiple event types with different HRFs

Key takeaways:
- The formula interface provides an intuitive way to specify models
- Different HRF choices affect model flexibility and interpretability
- Contrasts allow testing specific hypotheses about your data
- Complex designs can be built by combining multiple event models
- Always visualize your design matrix to verify the model structure